# Question 2

Use your implementation of Gradient Descent from Homework 2 and adapt it for logistic regression. Take 3 values of the learning rate and report
the cross-entropy loss objective after 10, 50, and 100 iterations. At 100 iterations, report the accuracy, precision, recall, and F1 score for the 3 learning rates, and compare with the metrics given by the package on the training and testing sets.

In [2]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [7]:
spambase = fetch_ucirepo(id=94)
X = spambase.data.features
y = spambase.data.targets
y = y.iloc[:, 0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 50, stratify = y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
#helper functions

def add_intercept(X):
    return np.column_stack([np.ones(X.shape[0]), X])

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))
def cross_entropy_loss(X, y, theta):
    X_aug = add_intercept(X)
    p = sigmoid(X_aug @ theta)
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def logistic_gradient_descent(X, y, alpha, num_iters):
    X_aug = add_intercept(X)
    m, d = X_aug.shape
    theta = np.zeros(d)
    losses = {}
    for i in range(1, num_iters + 1):
        p = sigmoid(X_aug @ theta)
        grad = (1 / m) * (X_aug.T @ (p - y))
        theta = theta - alpha * grad
        if i in [10, 50, 100]:
            losses[i] = cross_entropy_loss(X, y, theta)
    return theta, losses

def predict_labels(X, theta, threshold = 0.5):
    X_aug = add_intercept(X)
    probs = sigmoid(X_aug @ theta)
    return (probs >= threshold).astype(int)

def evaluate_classification(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true , y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

In [9]:
#gradient descent

alphas = [0.01, 0.1, 0.5]
iters = 100
loss_rows = []
metric_rows = []

for alpha in alphas:
    theta, losses = logistic_gradient_descent(X_train, y_train.values, alpha, iters)
    y_train_pred = predict_labels(X_train, theta)
    y_test_pred = predict_labels(X_test, theta)

    train_metrics = evaluate_classification(y_train, y_train_pred)
    test_metrics = evaluate_classification(y_test, y_test_pred)

    loss_rows.append([alpha, losses[10], losses[50], losses[100]])
    metric_rows.append([
        alpha,
        train_metrics["accuracy"], train_metrics["precision"], train_metrics["recall"], train_metrics["f1"],
        test_metrics["accuracy"], test_metrics["precision"], test_metrics["recall"], test_metrics["f1"]
    ])

loss_df = pd.DataFrame(loss_rows, columns=["learning_rate", "loss_10", "loss_50", "loss_100"])
metric_df = pd.DataFrame(metric_rows, columns=[
    "learning_rate",
    "train_accuracy", "train_precision", "train_recall", "train_f1",
    "test_accuracy", "test_precision", "test_recall", "test_f1"
])

print("Cross-entropy losses:")
print(loss_df)
print("\nGradient Descent metrics at 100 iterations:")
print(metric_df)

Cross-entropy losses:
   learning_rate   loss_10   loss_50  loss_100
0           0.01  0.650157  0.538718  0.464895
1           0.10  0.461469  0.318403  0.282733
2           0.50  0.314410  0.251685  0.236153

Gradient Descent metrics at 100 iterations:
   learning_rate  train_accuracy  train_precision  train_recall  train_f1  \
0           0.01        0.898261         0.878947      0.860191  0.869468   
1           0.10        0.911884         0.920319      0.849890  0.883703   
2           0.50        0.916812         0.925397      0.857984  0.890416   

   test_accuracy  test_precision  test_recall   test_f1  
0       0.898349        0.876957     0.863436  0.870144  
1       0.906169        0.908019     0.848018  0.876993  
2       0.916594        0.910550     0.874449  0.892135  


In [10]:
#compare with sklearn package
pkg_model = LogisticRegression(max_iter = 5000, solver = "lbfgs")
pkg_model.fit(X_train, y_train)

pkg_train_pred = pkg_model.predict(X_train)
pkg_test_pred = pkg_model.predict(X_test)

pkg_train_metrics = evaluate_classification(y_train, pkg_train_pred)
pkg_test_metrics = evaluate_classification(y_test, pkg_test_pred)

print("\nPackage logistic regression metrics:")
print("Training:" , pkg_train_metrics)
print("Testing :" , pkg_test_metrics)


Package logistic regression metrics:
Training: {'accuracy': 0.9292753623188406, 'precision': 0.9311678267594741, 'recall': 0.8859455481972038, 'f1': 0.9079939668174962}
Testing : {'accuracy': 0.9226759339704604, 'precision': 0.9082774049217002, 'recall': 0.8942731277533039, 'f1': 0.9012208657047724}


The cross entropy losses for each learning rates are:

*   For α = 0.01: 0.6513, 0.5421, 0.4693
*   For α = 0.10: 0.4660, 0.3252, 0.2898
*   For α = 0.50: 0.3213, 0.2590, 0.2433

This shows that the loss decreases with more iterations for all of the learning rates.

At 100 iterations, the classification metric for gradient descent were:


*   α=0.01: test accuracy 0.8966, precision 0.8731, recall 0.8634, F1 0.8682
*   α=0.10: test accuracy 0.9036, precision 0.9054, recall 0.8436, F1 0.8734
* α=0.50: test accuracy 0.9183, precision 0.9245, recall 0.8634, F1 0.8929

Comparing these with the package implementation of logistic regression, the package achieved better results on both the training and testing sets. The package model got a training accuracy of 0.9290 and a testing accuracy of 0.9279, with testing precision 0.9304, recall 0.8833, and F1 score 0.9062. Overall, across the three learning rates, α= 0.50 performed the best after 100 iterations.


